# Don Bossing — Free ComfyUI (Kaggle P100)

Runs ComfyUI + CogVideoX / HunyuanVideo on a **free Kaggle GPU** (P100 16GB, ~30 GPU-hrs/week) and exposes it through a public `cloudflared` tunnel so your local `server.js` can drive it.

Steps:
1. Set this notebook's accelerator to **GPU** (Settings → Accelerator → GPU P100).
2. Run the cells top to bottom.
3. Copy the `https://...trycloudflare.com` URL printed at the end into `video.config.json` (`comfyUrl`) on your PC.
4. On your PC run `npm run local`, open http://localhost:3000, and use Section 6.

When the session expires, just re-run and update the URL.

In [ ]:
# --- 1. Install ComfyUI + wrappers ---
!git clone https://github.com/comfyanonymous/ComfyUI
%cd ComfyUI
!pip install -q -r requirements.txt
!git clone https://github.com/kijai/ComfyUI-CogVideoXWrapper custom_nodes/ComfyUI-CogVideoXWrapper
!git clone https://github.com/kijai/ComfyUI-HunyuanVideoWrapper custom_nodes/ComfyUI-HunyuanVideoWrapper
!git clone https://github.com/comfyanonymous/ComfyUI-VideoHelperSuite custom_nodes/ComfyUI-VideoHelperSuite
!pip install -q -r custom_nodes/ComfyUI-CogVideoXWrapper/requirements.txt 2>/dev/null
!pip install -q -r custom_nodes/ComfyUI-HunyuanVideoWrapper/requirements.txt 2>/dev/null
print('comfyui + wrappers installed')

In [ ]:
# --- 2. Download model weights (only CogVideoX fits the free 16GB P100) ---
# CogVideoX 5B I2V (quantized). Needs ~5-7 GB VRAM.
!mkdir -p models/CogVideoX models/text_encoders
!hf_download() { huggingface-cli download "$1" "$2" --local-dir "$3" --local-dir-use-symlinks False; }
!huggingface-cli download THUDM/CogVideoX-5b-I2V --local-dir models/CogVideoX --local-dir-use-symlinks False
!huggingface-cli download comfyanonymous/flux_text_encoders t5xxl_fp8_e4m3fn.safetensors --local-dir models/text_encoders --local-dir-use-symlinks False

# Optional: HunyuanVideo I2V (needs ~24GB — will likely FAIL on free P100; server.js auto-disables it).
# !mkdir -p models/diffusion_models models/clip_vision models/vae models/text_encoders
# !huggingface-cli download Comfy-Org/HunyuanVideo_repackaged hunyuan_video_v2_replace_image_to_video_720p_bf16.safetensors --local-dir models/diffusion_models --local-dir-use-symlinks False
# !huggingface-cli download Comfy-Org/HunyuanVideo_repackaged clip_l.safetensors llava_llama3_fp8_scaled.safetensors --local-dir models/text_encoders --local-dir-use-symlinks False
# !huggingface-cli download Comfy-Org/HunyuanVideo_repackaged llava_llama3_vision.safetensors --local-dir models/clip_vision --local-dir-use-symlinks False
# !huggingface-cli download Comfy-Org/HunyuanVideo_repackaged hunyuan_video_vae_bf16.safetensors --local-dir models/vae --local-dir-use-symlinks False
print('weights ready')

In [ ]:
# --- 3. Launch ComfyUI in the background ---
import subprocess, os, time
log = open('comfy.log','w')
proc = subprocess.Popen(['python','main.py','--listen','0.0.0.0','--port','8188','--disable-auto-launch'],
                        stdout=log, stderr=subprocess.STDOUT)
print('ComfyUI launching (pid', proc.pid, ')...')
time.sleep(25)
print(open('comfy.log').read()[-1500:])

In [ ]:
# --- 4. Expose ComfyUI via a free cloudflared tunnel (no signup) ---
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
tun = subprocess.Popen(['./cloudflared','tunnel','--url','http://localhost:8188'],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
import re, sys
url = None
for line in tun.stdout:
    print(line.rstrip())
    m = re.search(r'(https://[a-z0-9\-]+\.trycloudflare\.com)', line)
    if m and not url:
        url = m.group(1)
        print('\n=== TUNNEL URL (paste into video.config.json comfyUrl) ===')
        print(url)
        print('=========================================================')
        break
print('Tunnel URL:', url)

In [ ]:
# --- 5. Keep-alive: poll ComfyUI so the session stays busy ---
import urllib.request, time, json
while True:
    try:
        with urllib.request.urlopen('http://localhost:8188/system_stats', timeout=5) as r:
            s = json.load(r)
            print('ComfyUI alive. GPU:', s.get('devices',[{}])[0].get('name','?'))
    except Exception as e:
        print('ComfyUI check:', e)
    time.sleep(60)